# Feature Engineering from BigQuery

This notebook loads the merged GridZero dataset from BigQuery, engineers lag and calendar features, validates the output, and saves the final modelling dataset.

In [ ]:
# %pip install google-cloud-bigquery db-dtypes


# Import Libraries

In [23]:
import pandas as pd
from google.cloud import bigquery

## Connect to BigQuery
Set the Google Cloud Project and Create a BigQuery client.

In [24]:
PROJECT_ID = "gridzero-489711"
DATASET = "merged_set"
TABLE = "Fully_merged_dataset_2025"

client = bigquery.Client(project=PROJECT_ID)

## Preview the Merged Dataset
To inspect the table structure.

In [25]:
query_preview = f"""
SELECT *
FROM `{PROJECT_ID}.{DATASET}.{TABLE}`
LIMIT 10
"""

df_preview = client.query(query_preview).to_dataframe()
df_preview.head()

/home/dd4real2k/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,time,temperature_2m_c,wind_speed_100m_ms,wind_gusts_10m_ms,cloud_cover_pct,shortwave_radiation_wm2,direct_radiation_wm2,diffuse_radiation_wm2,pressure_msl_hpa,snowfall_cm,...,Fossil Oil,Hydro Pumped Storage,Hydro Run-of-river and poundage,Nuclear,Other,Solar,Wind Offshore,Wind Onshore,TotalOutput-MW,carbon_intensity
0,2025-01-01 00:00:00+00:00,10.5,48.0,55.1,100,0.0,0.0,0.0,1014.2,0.0,...,0.0,0.0,736.0,5065.0,183.0,0.0,11444.531,9113.028,31028.559,55
1,2025-01-01 00:30:00+00:00,10.5,48.0,55.1,100,0.0,0.0,0.0,1014.2,0.0,...,0.0,0.0,745.0,5063.0,290.0,0.0,11138.565,8969.868,31138.433,54
2,2025-01-01 01:00:00+00:00,11.1,46.3,58.3,100,0.0,0.0,0.0,1013.1,0.0,...,0.0,0.0,746.0,5056.0,333.0,0.0,10788.770,8931.922,30826.692,53
3,2025-01-01 01:30:00+00:00,11.1,46.3,58.3,100,0.0,0.0,0.0,1013.1,0.0,...,0.0,0.0,746.0,5060.0,238.0,0.0,10519.534,8810.976,30209.510,53
4,2025-01-01 02:00:00+00:00,11.1,43.6,60.5,100,0.0,0.0,0.0,1012.0,0.0,...,0.0,0.0,747.0,5056.0,277.0,0.0,10706.056,8456.386,29988.442,47


In [26]:
# Inspect the available columns before feature engineering.
df_preview.columns.tolist()

['time',
 'temperature_2m_c',
 'wind_speed_100m_ms',
 'wind_gusts_10m_ms',
 'cloud_cover_pct',
 'shortwave_radiation_wm2',
 'direct_radiation_wm2',
 'diffuse_radiation_wm2',
 'pressure_msl_hpa',
 'snowfall_cm',
 'Biomass',
 'Fossil Gas',
 'Fossil Hard coal',
 'Fossil Oil',
 'Hydro Pumped Storage',
 'Hydro Run-of-river and poundage',
 'Nuclear',
 'Other',
 'Solar',
 'Wind Offshore',
 'Wind Onshore',
 'TotalOutput-MW',
 'carbon_intensity']

## Load the full Dataset
Load the full merged modelling dataset from BigQuery.

In [27]:
query_full = f"""
SELECT *
FROM `{PROJECT_ID}.{DATASET}.{TABLE}`
ORDER BY time
"""

df = client.query(query_full).to_dataframe()
df.head()

/home/dd4real2k/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,time,temperature_2m_c,wind_speed_100m_ms,wind_gusts_10m_ms,cloud_cover_pct,shortwave_radiation_wm2,direct_radiation_wm2,diffuse_radiation_wm2,pressure_msl_hpa,snowfall_cm,...,Fossil Oil,Hydro Pumped Storage,Hydro Run-of-river and poundage,Nuclear,Other,Solar,Wind Offshore,Wind Onshore,TotalOutput-MW,carbon_intensity
0,2025-01-01 00:00:00+00:00,10.5,48.0,55.1,100,0.0,0.0,0.0,1014.2,0.0,...,0.0,0.0,736.0,5065.0,183.0,0.0,11444.531,9113.028,31028.559,55
1,2025-01-01 00:30:00+00:00,10.5,48.0,55.1,100,0.0,0.0,0.0,1014.2,0.0,...,0.0,0.0,745.0,5063.0,290.0,0.0,11138.565,8969.868,31138.433,54
2,2025-01-01 01:00:00+00:00,11.1,46.3,58.3,100,0.0,0.0,0.0,1013.1,0.0,...,0.0,0.0,746.0,5056.0,333.0,0.0,10788.770,8931.922,30826.692,53
3,2025-01-01 01:30:00+00:00,11.1,46.3,58.3,100,0.0,0.0,0.0,1013.1,0.0,...,0.0,0.0,746.0,5060.0,238.0,0.0,10519.534,8810.976,30209.510,53
4,2025-01-01 02:00:00+00:00,11.1,43.6,60.5,100,0.0,0.0,0.0,1012.0,0.0,...,0.0,0.0,747.0,5056.0,277.0,0.0,10706.056,8456.386,29988.442,47


## Rename and clean the timestamp column
Rename `time` to `datetime` and convert it to datetime format.

In [28]:
df = df.rename(columns={"time": "datetime"})
df["datetime"] = pd.to_datetime(df["datetime"])
df.head

<bound method NDFrame.head of                        datetime  temperature_2m_c  wind_speed_100m_ms  \
0     2025-01-01 00:00:00+00:00              10.5                48.0   
1     2025-01-01 00:30:00+00:00              10.5                48.0   
2     2025-01-01 01:00:00+00:00              11.1                46.3   
3     2025-01-01 01:30:00+00:00              11.1                46.3   
4     2025-01-01 02:00:00+00:00              11.1                43.6   
...                         ...               ...                 ...   
17514 2025-12-31 21:00:00+00:00              -0.8                22.5   
17515 2025-12-31 21:30:00+00:00              -0.8                22.5   
17516 2025-12-31 22:00:00+00:00              -0.9                23.3   
17517 2025-12-31 22:30:00+00:00              -0.9                23.3   
17518 2025-12-31 23:00:00+00:00              -0.8                23.4   

       wind_gusts_10m_ms  cloud_cover_pct  shortwave_radiation_wm2  \
0                   55.

In [31]:
df["datetime"].duplicated().sum()

np.int64(0)

In [32]:
df.shape

(17519, 23)

## Make the project root importable
This allows the notebook to import reusable functions from `python_scripts`.

In [33]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

print("Project root added to path:", project_root)

Project root added to path: /home/dd4real2k/code/mlbh/GridZero


# Import reusable feature engineering functions

In [34]:
from python_scripts.feature_engineering import (
    engineer_features,
    validate_features,
    drop_lag_nulls
)

## Engineer lag and calendar features
Create lag features from `carbon_intensity` and add calendar-based predictors.

In [35]:
df = engineer_features(df, target_col="carbon_intensity", add_year_lag=False)
df.head()

,datetime,temperature_2m_c,wind_speed_100m_ms,wind_gusts_10m_ms,cloud_cover_pct,shortwave_radiation_wm2,direct_radiation_wm2,diffuse_radiation_wm2,pressure_msl_hpa,snowfall_cm,...,Wind Offshore,Wind Onshore,TotalOutput-MW,carbon_intensity,lag_48,lag_336,hour,day_of_week,month,is_weekend
0,2025-01-01 00:00:00+00:00,10.5,48.0,55.1,100,0.0,0.0,0.0,1014.2,0.0,...,11444.531,9113.028,31028.559,55,<NA>,<NA>,0,2,1,0
1,2025-01-01 00:30:00+00:00,10.5,48.0,55.1,100,0.0,0.0,0.0,1014.2,0.0,...,11138.565,8969.868,31138.433,54,<NA>,<NA>,0,2,1,0
2,2025-01-01 01:00:00+00:00,11.1,46.3,58.3,100,0.0,0.0,0.0,1013.1,0.0,...,10788.770,8931.922,30826.692,53,<NA>,<NA>,1,2,1,0
3,2025-01-01 01:30:00+00:00,11.1,46.3,58.3,100,0.0,0.0,0.0,1013.1,0.0,...,10519.534,8810.976,30209.510,53,<NA>,<NA>,1,2,1,0
4,2025-01-01 02:00:00+00:00,11.1,43.6,60.5,100,0.0,0.0,0.0,1012.0,0.0,...,10706.056,8456.386,29988.442,47,<NA>,<NA>,2,2,1,0


## Check missing values after lag creation
Lag features create missing values at the start of the dataset.

In [36]:
df.isnull().sum()

datetime                             0
temperature_2m_c                     0
wind_speed_100m_ms                   0
wind_gusts_10m_ms                    0
cloud_cover_pct                      0
shortwave_radiation_wm2              0
direct_radiation_wm2                 0
diffuse_radiation_wm2                0
pressure_msl_hpa                     0
snowfall_cm                          0
Biomass                             45
Fossil Gas                          45
Fossil Hard coal                    45
Fossil Oil                          45
Hydro Pumped Storage                45
Hydro Run-of-river and poundage     45
Nuclear                             45
Other                               45
Solar                               45
Wind Offshore                       45
Wind Onshore                        45
TotalOutput-MW                      45
carbon_intensity                    47
lag_48                              48
lag_336                            336
hour                     

## Drop rows with lag-related missing values
Remove incomplete rows before modelling.

In [37]:
df = drop_lag_nulls(df)
df.head()

,datetime,temperature_2m_c,wind_speed_100m_ms,wind_gusts_10m_ms,cloud_cover_pct,shortwave_radiation_wm2,direct_radiation_wm2,diffuse_radiation_wm2,pressure_msl_hpa,snowfall_cm,...,Wind Offshore,Wind Onshore,TotalOutput-MW,carbon_intensity,lag_48,lag_336,hour,day_of_week,month,is_weekend
0,2025-01-08 00:00:00+00:00,0.4,19.9,20.9,0,0.0,0.0,0.0,1003.6,0.0,...,10038.334,5580.935,28845.269,86,69,55,0,2,1,0
1,2025-01-08 00:30:00+00:00,0.4,19.9,20.9,0,0.0,0.0,0.0,1003.6,0.0,...,9821.886,5547.003,28739.889,84,68,54,0,2,1,0
2,2025-01-08 01:00:00+00:00,0.6,18.9,18.7,13,0.0,0.0,0.0,1003.9,0.0,...,9673.525,5395.306,28531.831,85,65,53,1,2,1,0
3,2025-01-08 01:30:00+00:00,0.6,18.9,18.7,13,0.0,0.0,0.0,1003.9,0.0,...,9508.819,5288.912,27985.731,88,66,53,1,2,1,0
4,2025-01-08 02:00:00+00:00,0.2,19.2,18.4,97,0.0,0.0,0.0,1004.4,0.0,...,9436.926,4880.031,27592.957,90,63,47,2,2,1,0


## Validate the engineered dataset
Run checks on shape, columns, missing values, duplicates, and datetime spacing.

In [38]:
validate_features(df)

Shape: (17092, 29)

Columns:
['datetime', 'temperature_2m_c', 'wind_speed_100m_ms', 'wind_gusts_10m_ms', 'cloud_cover_pct', 'shortwave_radiation_wm2', 'direct_radiation_wm2', 'diffuse_radiation_wm2', 'pressure_msl_hpa', 'snowfall_cm', 'Biomass', 'Fossil Gas', 'Fossil Hard coal', 'Fossil Oil', 'Hydro Pumped Storage', 'Hydro Run-of-river and poundage', 'Nuclear', 'Other', 'Solar', 'Wind Offshore', 'Wind Onshore', 'TotalOutput-MW', 'carbon_intensity', 'lag_48', 'lag_336', 'hour', 'day_of_week', 'month', 'is_weekend']

Missing values:
datetime                           0
temperature_2m_c                   0
wind_speed_100m_ms                 0
wind_gusts_10m_ms                  0
cloud_cover_pct                    0
shortwave_radiation_wm2            0
direct_radiation_wm2               0
diffuse_radiation_wm2              0
pressure_msl_hpa                   0
snowfall_cm                        0
Biomass                            0
Fossil Gas                         0
Fossil Hard coal   

## Clean column names for modelling
Replace spaces and hyphens in column names to make them easier to use in modelling.

In [39]:
df.columns = (
    df.columns
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
    .str.replace("/", "_", regex=False)
)

df.columns.tolist()

['datetime',
 'temperature_2m_c',
 'wind_speed_100m_ms',
 'wind_gusts_10m_ms',
 'cloud_cover_pct',
 'shortwave_radiation_wm2',
 'direct_radiation_wm2',
 'diffuse_radiation_wm2',
 'pressure_msl_hpa',
 'snowfall_cm',
 'Biomass',
 'Fossil_Gas',
 'Fossil_Hard_coal',
 'Fossil_Oil',
 'Hydro_Pumped_Storage',
 'Hydro_Run_of_river_and_poundage',
 'Nuclear',
 'Other',
 'Solar',
 'Wind_Offshore',
 'Wind_Onshore',
 'TotalOutput_MW',
 'carbon_intensity',
 'lag_48',
 'lag_336',
 'hour',
 'day_of_week',
 'month',
 'is_weekend']

## Save the feature-engineered dataset locally
Export the final modelling dataset as a CSV file.

In [16]:
# df.to_csv("../data/model_dataset_2025.csv", index=False)

In [42]:
# df.to_csv("../notebooks/data/model_dataset_2025.csv", index=False)

In [17]:
table_id = "gridzero-489711.merged_set.feature_engineered_dataset_2025"

job = client.load_table_from_dataframe(df, table_id)

job.result()

print("Upload complete")

/home/dd4real2k/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Upload complete


In [45]:
# Verify the uploaded feature-engineered table

check_query = f"""
SELECT COUNT(*) AS row_count
FROM `{PROJECT_ID}.{DATASET}.feature_engineered_dataset_2025`
"""

client.query(check_query).to_dataframe()

/home/dd4real2k/.pyenv/versions/3.12.9/envs/gridzero/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,row_count
0,17092


In [47]:
## Preview the final modelling dataset
df.head()

,datetime,temperature_2m_c,wind_speed_100m_ms,wind_gusts_10m_ms,cloud_cover_pct,shortwave_radiation_wm2,direct_radiation_wm2,diffuse_radiation_wm2,pressure_msl_hpa,snowfall_cm,...,Wind_Offshore,Wind_Onshore,TotalOutput_MW,carbon_intensity,lag_48,lag_336,hour,day_of_week,month,is_weekend
0,2025-01-08 00:00:00+00:00,0.4,19.9,20.9,0,0.0,0.0,0.0,1003.6,0.0,...,10038.334,5580.935,28845.269,86,69,55,0,2,1,0
1,2025-01-08 00:30:00+00:00,0.4,19.9,20.9,0,0.0,0.0,0.0,1003.6,0.0,...,9821.886,5547.003,28739.889,84,68,54,0,2,1,0
2,2025-01-08 01:00:00+00:00,0.6,18.9,18.7,13,0.0,0.0,0.0,1003.9,0.0,...,9673.525,5395.306,28531.831,85,65,53,1,2,1,0
3,2025-01-08 01:30:00+00:00,0.6,18.9,18.7,13,0.0,0.0,0.0,1003.9,0.0,...,9508.819,5288.912,27985.731,88,66,53,1,2,1,0
4,2025-01-08 02:00:00+00:00,0.2,19.2,18.4,97,0.0,0.0,0.0,1004.4,0.0,...,9436.926,4880.031,27592.957,90,63,47,2,2,1,0
